# Vectora — embedding models at passage level (CPU only)

**Question:** does a better embedding model find the *passage* more often? Run 2 said "no difference" at **paper** level on 88 questions; run 3 showed passage level is where Vectora struggles (0.258). This re-tests it properly, on 365 questions, with no GPU and no LLM.

**One variable:** the chunks are built once (MiniLM semantic chunking) and **re-used unchanged** for every model, so only the embedding model differs.

| variant | model | dims |
|---|---|---|
| baseline | all-MiniLM-L6-v2 | 384 |
| bge-small | BAAI/bge-small-en-v1.5 | 384 |
| bge-base | BAAI/bge-base-en-v1.5 | 768 |

Each is scored in **fast** mode and **balanced --no-llm** (reranker on), and calibrated, because BGE scores sit higher and need their own refusal floor (run 2 lesson).
Note: BGE is trained with a short instruction before queries, which Vectora does not add, so BGE is slightly understated — a BGE win is real, a BGE loss may not be.

**Runtime:** CPU (Runtime → Change runtime type → **CPU**). Run cell 1 alone, approve Drive, then Run all and keep the tab open. About 2–2.5 hours. Results save to `MyDrive/vectora_results/<run>/` as each variant finishes.

## 1. Drive

In [ ]:
import datetime, os
RUN = "embed_" + datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
from google.colab import drive
drive.mount("/content/drive")
OUT = f"/content/drive/MyDrive/vectora_results/{RUN}"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT)

## 2. Vectora, no LLM

In [ ]:
!git clone -q https://github.com/SiddharthaRaja/Vectora.git /content/Vectora
%cd /content/Vectora
!git log --oneline -1 | tee {OUT}/commit.txt
!pip install -q -e ".[ingest,eval]" 2>&1 | grep -v -i "warning\|dependency resolver\|incompatible" | tail -2
open(".env", "w").write("""
VECTORA_CORPUS=qasper-300
VECTORA_QDRANT_URL=
VECTORA_QDRANT_PATH=./data/qdrant
VECTORA_MIN_DENSE_SCORE=0.40
VECTORA_MIN_RERANK_SCORE=-4.0
VECTORA_CORS_ORIGINS=http://localhost:5000
VECTORA_LOG_LEVEL=WARNING
""")

## 3. Build the corpus and chunk it once (MiniLM semantic chunking)

In [ ]:
!python scripts/qasper_corpus.py --corpus qasper-300 --split all --papers 300 --force 2>&1 | tail -4
!date; vectora --corpus qasper-300 ingest 2>&1 | grep -E "chunks emitted|papers processed|failed"; date

## 4. One corpus per embedding model, same chunks

Each variant is its own corpus (own Qdrant collection) holding a copy of the **same** `chunks.jsonl` and golden set, indexed with a different model.

In [ ]:
import shutil, subprocess, pathlib, json
VARIANTS = {
    "baseline":  {"VECTORA_EMBEDDING_MODEL": "sentence-transformers/all-MiniLM-L6-v2", "VECTORA_EMBEDDING_DIM": "384"},
    "bge-small": {"VECTORA_EMBEDDING_MODEL": "BAAI/bge-small-en-v1.5", "VECTORA_EMBEDDING_DIM": "384"},
    "bge-base":  {"VECTORA_EMBEDDING_MODEL": "BAAI/bge-base-en-v1.5",  "VECTORA_EMBEDDING_DIM": "768"},
}
src = pathlib.Path("corpora/qasper-300")

def sh(cmd, env, log):
    with open(log, "a") as f:
        f.write(f"\n$ {cmd}\n"); f.flush()
        return subprocess.run(cmd, shell=True, env=env, stdout=f, stderr=subprocess.STDOUT).returncode

for name, overrides in VARIANTS.items():
    corpus = f"q300-{name}"
    env = {**os.environ, **overrides}
    log = f"{OUT}/{name}.log"
    print("=====", name, overrides["VECTORA_EMBEDDING_MODEL"]); subprocess.run("date", shell=True)
    sh(f'vectora corpus create {corpus} -d "qasper-300, {name} embeddings" --kind "NLP research papers"', env, log)
    d = pathlib.Path("corpora") / corpus
    for f in ("chunks.jsonl", "golden.yaml"):
        shutil.copy(src / f, d / f)
    sh(f"vectora --corpus {corpus} index --recreate", env, log)
    sh(f"vectora --corpus {corpus} eval --mode fast --no-llm --output {OUT}/{name}_fast.json", env, log)
    sh(f"vectora --corpus {corpus} eval --mode balanced --no-llm --output {OUT}/{name}_balanced.json", env, log)
    sh(f"vectora --corpus {corpus} calibrate --mode fast", env, log)
    for mode in ("fast", "balanced"):
        try:
            r = json.load(open(f"{OUT}/{name}_{mode}.json"))["reports"][0]
            print(f"  {mode:<9} passage {r['recall']:.3f}  paper {r['paper_recall']:.3f}  MRR {r['mrr']:.3f}  neg ok {r['negative_rejection_rate']:.0%}  p50 {r['median_latency_ms']:.0f} ms")
        except Exception as e:
            print(f"  {mode}: no result ({e}) — see {log}")

## 5. Paired comparison against the baseline (same questions)

In [ ]:
import random
def misses(name, mode):
    r = json.load(open(f"{OUT}/{name}_{mode}.json"))["reports"][0]
    return set(r["misses"]), r["cases"]
rows = []
for mode in ("fast", "balanced"):
    base, n = misses("baseline", mode)
    for name in ("bge-small", "bge-base"):
        try:
            other, _ = misses(name, mode)
        except Exception:
            continue
        fixed, broken = len(base - other), len(other - base)
        diff = [1] * fixed + [-1] * broken + [0] * (n - fixed - broken)
        random.seed(0)
        boots = sorted(sum(random.choice(diff) for _ in range(n)) / n for _ in range(4000))
        line = f"{mode:<9} {name:<10} passage recall {((fixed - broken) / n):+.3f}  (fixed {fixed}, broke {broken})  95% CI {boots[100]:+.3f} to {boots[3899]:+.3f}"
        print(line); rows.append(line)
open(f"{OUT}/paired.txt", "w").write("\n".join(rows) + "\n")
!date | tee {OUT}/finished_at.txt
!ls -la {OUT}